###  Package

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import os

os.getcwd()
# os.chdir("/home/bardia/graphical-sampling")
# os.chdir("/home/bardia/graphical-sampling/simulations")
os.chdir('/config/ws/graphical-sampling/simulations')
# os.chdir('/Users/Apc/graphical-sampling/simulations')

In [ ]:
# Loading the data and preparing the DataFrames for sampling

import pandas as pd
import numpy as np

# Assuming inclusion_probabilities is defined elsewhere or imported
# from your package
files = ["aggregated.csv", "regular.csv", "meuse.csv", "swiss.csv", "aggregated_1000.csv", "regular_1000.csv",
         "AggregatedPop1000.csv", "RegularPop1000.csv"]

def popus(n, path="populations/real"):
    dfs = {} # Better to define inside to keep it clean
    for f in files:
        full_path = os.path.join(path, f)
        if not os.path.exists(full_path):
            print(f"Warning: {full_path} not found.")
            continue
            
        df = pd.read_csv(full_path)
        N_pop = len(df)
        name = f.replace('.csv', '')
        
        # Equal Probabilities (Standard for all)
        pik_ep = np.repeat(n / N_pop, N_pop)
        
        if f in ["aggregated.csv", "regular.csv",  "aggregated_1000.csv", "regular_1000.csv",
                  "AggregatedPop1000.csv", "RegularPop1000.csv"]:
            coords = df.to_numpy()
            dfs[f'df_{name}'] = pd.DataFrame({
                'coord_x': coords[:, 0],
                'coord_y': coords[:, 1],
                'pik_ep' : pik_ep,
                'z'      : np.ones(N_pop) # Placeholder if z is missing
            })

        elif f == "meuse.csv":
            coords = df[['x', 'y']].to_numpy()
            # Unequal Probabilities (UP)
            pik_up = inclusion_probabilities(df['copper'].to_numpy().copy(), n)
            dfs[f'df_{name}'] = pd.DataFrame({
                'coord_x': coords[:, 0],
                'coord_y': coords[:, 1],
                'pik_ep' : pik_ep,
                'pik_up' : pik_up,
                'z'      : df['cadmium']
            })

        elif f == "swiss.csv":
            coords = df[['x', 'y']].to_numpy()
            AREA = df['AREA'].to_numpy().clip(5, 100)
            pik_up = inclusion_probabilities(AREA.copy(), n)
            dfs[f'df_{name}'] = pd.DataFrame({
                'coord_x': coords[:, 0],
                'coord_y': coords[:, 1],
                'pik_ep' : pik_ep,
                'pik_up' : pik_up,
                'z'      : df['AREA_A']
            })
            
    return dfs

In [10]:
# --- Parameters ---
popu_regular = 'df_RegularPop1000'
popu_aggregated = 'df_AggregatedPop1000'
n=50
df_regular = popus(n)[popu_regular]
df_aggregated = popus(n)[popu_aggregated]

N = len(df_regular)
df_regular[['x1', 'x2']]=df_regular[['coord_x', 'coord_y']]
df_aggregated[['x1', 'x2']]=df_aggregated[['coord_x', 'coord_y']]

coords_data_regular = df_regular[['x1', 'x2']].to_numpy()
coords_data_aggregated = df_aggregated[['x1', 'x2']].to_numpy()


In [17]:
import numpy as np

# =========================================================
# SHORTCUTS
# =========================================================

x1r = df_regular["x1"].to_numpy()
x2r = df_regular["x2"].to_numpy()

x1a = df_aggregated["x1"].to_numpy()
x2a = df_aggregated["x2"].to_numpy()

# =========================================================
# 1. CORRUGATED PLANE
# =========================================================

df_regular["corrugated"] = (
    3*(x1r + x2r)
    +
    np.sin(6*(x1r + x2r))
)

df_aggregated["corrugated"] = (
    3*(x1a + x2a)
    +
    np.sin(6*(x1a + x2a))
)

# =========================================================
# 2. PEAK FUNCTION
# =========================================================

def peak_function(x1, x2):

    part1 = (
        3*(4 - 6*x1)**2
        *
        np.exp(
            -(6*x1 - 3)**2
            -
            (6*x2 - 2)**2
        )
    )

    part2 = (
        -10
        *
        (
            0.2*(6*x1 - 3)
            -
            (6*x1 - 3)**3
            -
            (6*x2 - 3)**5
        )
        *
        np.exp(
            -(6*x1 - 3)**2
            -
            (6*x2 - 3)**2
        )
    )

    part3 = (
        -(1/3)
        *
        np.exp(
            -(6*x1 - 2)**2
            -
            (6*x2 - 3)**2
        )
    )

    return 20*(part1 + part2 + part3)

df_regular["peak"] = peak_function(x1r, x2r)

df_aggregated["peak"] = peak_function(x1a, x2a)

# =========================================================
# 3. BIRD FUNCTION
# =========================================================

def bird_function(x1, x2):

    return (

        6/20

        * (

            (12*x1 - 12*x2)**2

            +

            np.exp(
                (1 - np.sin(12*x1 - 6))**2
            )
            *
            np.cos(12*x2 - 6)

            +

            np.exp(
                (1 - np.cos(12*x2 - 6))**2
            )
            *
            np.sin(12*x1 - 6)

        )
    )

df_regular["bird"] = bird_function(x1r, x2r)

df_aggregated["bird"] = bird_function(x1a, x2a)

# =========================================================
# CHECK POPULATION TOTALS
# =========================================================

print("\n================================================")
print("REGULAR POPULATION TOTALS")
print("================================================")

print(
    "Corrugated tau =",
    df_regular["corrugated"].sum()
)

print(
    "Peak tau =",
    df_regular["peak"].sum()
)

print(
    "Bird tau =",
    df_regular["bird"].sum()
)

print("\n================================================")
print("AGGREGATED POPULATION TOTALS")
print("================================================")

print(
    "Corrugated tau =",
    df_aggregated["corrugated"].sum()
)

print(
    "Peak tau =",
    df_aggregated["peak"].sum()
)

print(
    "Bird tau =",
    df_aggregated["bird"].sum()
)


REGULAR POPULATION TOTALS
Corrugated tau = 2999.3156558692144
Peak tau = 7414.3387554199235
Bird tau = 7075.839932794384

AGGREGATED POPULATION TOTALS
Corrugated tau = 2890.7252931792573
Peak tau = 5273.999411700915
Bird tau = 4026.2265307644648


In [ ]:
# --- Parameters ---
popu_regular = 'df_RegularPop1000'
ep_or_up = 'pik_ep'
n = 20
df = popus(n)[popu]
df = rotate_population(df.copy(),0)
N = len(df)

coords_data = df[['coord_x', 'coord_y']].to_numpy()
inclusions_data = df[ep_or_up].to_numpy()

